In [ ]:
%%capture
%pip install -q torch torchvision torchaudio
%pip install -q opencv-python numpy matplotlib pillow tqdm
%pip install -q scikit-image
%pip install -q dlib
%pip install -q face-alignment
%pip install -q opencv-python tqdm


In [1]:
import cv2
import os
import numpy as np
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
from functools import partial


### Bước 1: Tiền xử lý
-	Sử dụng một bộ phát hiện khuôn mặt (dùng OpenCV frontal face cascade classifier như trong bài báo) để tìm vùng khuôn mặt trong mỗi ảnh.
-	Cắt vùng mặt đã phát hiện được.
-	Thay đổi kích thước (resize) tất cả các ảnh mặt đã cắt về kích thước 128 x 128. 
-   Lưu ảnh đã cắt ra một thư mục mới có tên "128_crop_dataset"



In [ ]:
# Đường dẫn 
INPUT_FOLDER = "CelabA_dataset/img_align_celeba/img_align_celeba"
OUTPUT_FOLDER = "128_crop_dataset"
CASCADE_PATH = cv2.data.haarcascades + "haarcascade_frontalface_default.xml" # type: ignore

In [ ]:
def process_single_image(filename, input_folder, output_folder, cascade_path):
    try:
        img_path = os.path.join(input_folder, filename)
        save_path = os.path.join(output_folder, filename)

        # 1. Đọc ảnh
        img = cv2.imread(img_path)
        if img is None:
            return "READ_ERROR"

        # 2. Chuyển sang Grayscale để detect nhanh hơn
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Khởi tạo CascadeClassifier bên trong hàm worker
        face_cascade = cv2.CascadeClassifier(cascade_path)

        # 3. Phát hiện khuôn mặt
        faces = face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=4,
            minSize=(50, 50),
            flags=cv2.CASCADE_SCALE_IMAGE
        )

        if len(faces) == 0:
            return "NO_FACE"

        # 4. Lấy khuôn mặt lớn nhất
        x, y, w, h = sorted(faces, key=lambda box: box[2] * box[3], reverse=True)[0]
        face_crop = img[y:y+h, x:x+w]

        if face_crop.size == 0:
            return "CROP_ERROR"

        # 5. Resize về 128x128
        face_resized = cv2.resize(face_crop, (128, 128), interpolation=cv2.INTER_AREA)

        # 6. Lưu ảnh
        cv2.imwrite(save_path, face_resized)
        return "SUCCESS"

    except Exception as e:
        return f"ERROR: {str(e)}"


In [4]:
def main_sequential():
    if not os.path.exists(OUTPUT_FOLDER):
        os.makedirs(OUTPUT_FOLDER)
        print(f"Đã tạo thư mục: {OUTPUT_FOLDER}")

    if not os.path.exists(INPUT_FOLDER):
        print(f"Lỗi: Không tìm thấy thư mục đầu vào: {INPUT_FOLDER}")
        return

    all_files = [f for f in os.listdir(INPUT_FOLDER) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    total_files = len(all_files)
    print(f"Tìm thấy {total_files} ảnh. Bắt đầu xử lý tuần tự...")

    worker_func = partial(process_single_image, 
                          input_folder=INPUT_FOLDER, 
                          output_folder=OUTPUT_FOLDER, 
                          cascade_path=CASCADE_PATH)


    results = [worker_func(f) for f in tqdm(all_files, total=total_files, unit="img")]


    success_count = results.count("SUCCESS")
    no_face_count = results.count("NO_FACE")
    errors = [r for r in results if r not in ("SUCCESS", "NO_FACE", "SKIPPED")]

    print("\n--- HOÀN THÀNH (TUẦN TỰ) ---")
    print(f"Thành công: {success_count}")
    print(f"Không tìm thấy mặt: {no_face_count}")
    print(f"Lỗi khác: {len(errors)}")



In [5]:
main_sequential()

Tìm thấy 130114 ảnh. Bắt đầu xử lý tuần tự...


100%|██████████| 130114/130114 [2:12:59<00:00, 16.31img/s] 



--- HOÀN THÀNH (TUẦN TỰ) ---
Thành công: 124599
Không tìm thấy mặt: 5515
Lỗi khác: 0


In [1]:
import os
import shutil
from tqdm import tqdm

# --- CẤU HÌNH ĐƯỜNG DẪN ---
# Đường dẫn 1: Thư mục ảnh gốc (Full CelebA dataset)
ORIGINAL_DIR = "CelabA_dataset/img_align_celeba/img_align_celeba"

# Đường dẫn 2: Thư mục ảnh đã cắt thành công (Kết quả của bước trước)
PROCESSED_DIR = "128_crop_dataset"

# Đường dẫn 3: Thư mục chứa các ảnh bị lỗi (Không tìm thấy mặt ở bước trước)
MISSING_DIR = "failed_detection_images"

def find_and_copy_missing_files():
    # 1. Tạo thư mục đích nếu chưa tồn tại
    if not os.path.exists(MISSING_DIR):
        os.makedirs(MISSING_DIR)
        print(f"Đã tạo thư mục chứa ảnh lỗi: {MISSING_DIR}")

    # 2. Kiểm tra thư mục nguồn
    if not os.path.exists(ORIGINAL_DIR) or not os.path.exists(PROCESSED_DIR):
        print("Lỗi: Không tìm thấy thư mục gốc hoặc thư mục đã xử lý.")
        return

    # 3. Lấy danh sách tất cả file trong thư mục gốc
    # Chúng ta duyệt từ thư mục gốc để đảm bảo không bỏ sót file nào (từ 000001 đến hết)
    print("Đang quét danh sách file...")
    all_files = sorted([f for f in os.listdir(ORIGINAL_DIR) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
    
    print(f"Tổng số ảnh gốc: {len(all_files)}")
    
    missing_count = 0
    copied_count = 0

    # 4. Duyệt và so sánh
    print("Đang kiểm tra và copy ảnh thiếu...")
    for filename in tqdm(all_files, unit="img"):
        
        # Tạo đường dẫn kiểm tra trong thư mục đã xử lý (Đường dẫn 2)
        processed_path = os.path.join(PROCESSED_DIR, filename)
        
        # Nếu KHÔNG tồn tại trong thư mục 2 -> Nghĩa là bước trước đã bỏ qua nó
        if not os.path.exists(processed_path):
            missing_count += 1
            
            # Đường dẫn nguồn (Đường dẫn 1)
            src_path = os.path.join(ORIGINAL_DIR, filename)
            # Đường dẫn đích (Đường dẫn 3)
            dst_path = os.path.join(MISSING_DIR, filename)
            
            try:
                # Copy file sang thư mục lỗi
                shutil.copy2(src_path, dst_path)
                copied_count += 1
            except Exception as e:
                print(f"Lỗi khi copy file {filename}: {e}")

    # 5. Thông báo kết quả
    print("\n--- HOÀN THÀNH ---")
    print(f"Tổng số ảnh bị thiếu ở bước trước: {missing_count}")
    print(f"Đã copy thành công sang '{MISSING_DIR}': {copied_count}")

if __name__ == "__main__":
    find_and_copy_missing_files()

Đang quét danh sách file...
Tổng số ảnh gốc: 202599
Đang kiểm tra và copy ảnh thiếu...


100%|██████████| 202599/202599 [04:07<00:00, 818.00img/s] 


--- HOÀN THÀNH ---
Tổng số ảnh bị thiếu ở bước trước: 8624
Đã copy thành công sang 'failed_detection_images': 8624


In [ ]:
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
from functools import partial

# --- CẤU HÌNH ---
# Bạn có thể trỏ INPUT_FOLDER vào thư mục "failed_detection_images" để xử lý lại các ảnh lỗi
INPUT_FOLDER = "failed_detection_images" 
OUTPUT_FOLDER = "128_crop_dataset_yunet"

# Link tải model: https://github.com/opencv/opencv_zoo/tree/master/models/face_detection_yunet
# Tải file 'face_detection_yunet_2023mar.onnx' và để cùng thư mục với script này
MODEL_PATH = "face_detection_yunet_2023mar.onnx" 

def process_with_yunet(filename, input_folder, output_folder, model_path):
    try:
        img_path = os.path.join(input_folder, filename)
        save_path = os.path.join(output_folder, filename)

        # Đọc ảnh
        img = cv2.imread(img_path)
        if img is None:
            return "READ_ERROR"
        
        h, w, _ = img.shape

        # KHỞI TẠO YUNET (Deep Learning based)
        # Cần khởi tạo bên trong worker process
        detector = cv2.FaceDetectorYN.create(
            model=model_path,
            config="",
            input_size=(w, h), # YuNet cần biết kích thước ảnh đầu vào
            score_threshold=0.6, # Độ tin cậy (0.6 là khá chắc chắn)
            nms_threshold=0.3,
            top_k=1
        )

        # Phát hiện khuôn mặt
        # faces trả về danh sách: [x, y, w, h, ...landmarks...]
        _, faces = detector.detect(img)

        if faces is None or len(faces) == 0:
            return "NO_FACE"

        # Lấy khuôn mặt có độ tin cậy cao nhất (thường là mặt đầu tiên do top_k=1)
        face = faces[0]
        x, y, w_box, h_box = map(int, face[:4])

        # Xử lý biên để không bị lỗi out of bound khi crop
        x = max(0, x)
        y = max(0, y)
        w_box = min(w - x, w_box)
        h_box = min(h - y, h_box)

        # Crop ảnh
        face_crop = img[y:y+h_box, x:x+w_box]

        if face_crop.size == 0:
            return "CROP_ERROR"

        # Resize
        face_resized = cv2.resize(face_crop, (128, 128), interpolation=cv2.INTER_AREA)

        # Lưu ảnh
        cv2.imwrite(save_path, face_resized)
        return "SUCCESS"

    except Exception as e:
        return f"ERROR: {str(e)}"

def main():
    if not os.path.exists(OUTPUT_FOLDER):
        os.makedirs(OUTPUT_FOLDER)
    
    if not os.path.exists(MODEL_PATH):
        print(f"LỖI: Không tìm thấy file model '{MODEL_PATH}'")
        print("Vui lòng tải tại: https://github.com/opencv/opencv_zoo/raw/master/models/face_detection_yunet/face_detection_yunet_2023mar.onnx")
        return

    all_files = [f for f in os.listdir(INPUT_FOLDER) if f.lower().endswith(('.jpg', '.png'))]
    
    if not all_files:
        print(f"Thư mục {INPUT_FOLDER} trống hoặc không tồn tại.")
        return

    print(f"Bắt đầu xử lý lại {len(all_files)} ảnh với YuNet...")

    num_processes = max(1, cpu_count() - 1)
    worker_func = partial(process_with_yunet, 
                          input_folder=INPUT_FOLDER, 
                          output_folder=OUTPUT_FOLDER, 
                          model_path=MODEL_PATH)

    with Pool(processes=num_processes) as pool:
        results = list(tqdm(pool.imap(worker_func, all_files), total=len(all_files)))

    print(f"Thành công: {results.count('SUCCESS')}")
    print(f"Vẫn không tìm thấy mặt: {results.count('NO_FACE')}")

if __name__ == '__main__':
    main()

Tìm thấy 8624 ảnh. Bắt đầu xử lý tuần tự...


KeyboardInterrupt: 